# VITS → B1 droid — proper fine-tune (with discriminator)

Our own training loop (`colab_train.ipynb`) lacked the **adversarial / discriminator loss** that real VITS needs, so it produced noise (Whisper WER 90–120%). This notebook uses the **official HuggingFace recipe** [`ylacombe/finetune-hf-vits`](https://github.com/ylacombe/finetune-hf-vits), which has the full objective: mel + KL + duration **+ discriminator + generator + feature-matching**.

- **Base:** `ylacombe/vits-ljs-with-discriminator` — the same LJS VITS as before, with a discriminator already attached (no conversion needed).
- **Data:** [`Dmi1tr13/ljspeech-b1`](https://huggingface.co/datasets/Dmi1tr13/ljspeech-b1) — the droid voice (columns `audio` + `text`).

We train in **two stages**: first a tiny subset (fast sanity check that the pipeline actually learns), then the full dataset. The last 500 clips are held out as the test set in both stages.

**Before running:** `Runtime → Change runtime type → GPU (T4/L4) → Save`.

## 1. Clone the official recipe

In [ ]:
!git clone https://github.com/ylacombe/finetune-hf-vits.git
%cd finetune-hf-vits

## 2. Install dependencies
The recipe's requirements + system `espeak-ng` (the VITS tokenizer needs it) + `whisper`/`jiwer` for our objective evaluation.

In [ ]:
!apt-get -qq install -y espeak-ng
!pip install -q -r requirements.txt
# Pin transformers to the recipe's era: requirements only says >=4.35.1, so Colab installs a
# version new enough to have dropped `send_example_telemetry` (which run_vits_finetuning.py imports)
# and to break the recipe's bundled utils.py. 4.44.2 has the API the recipe was written against.
!pip install -q "transformers==4.44.2" jiwer openai-whisper phonemizer
print('deps OK')

## 3. Build the monotonic alignment extension
VITS training needs the Cython `monotonic_align` (Monotonic Alignment Search). This compiles it in place.

In [ ]:
%cd monotonic_align
!mkdir -p monotonic_align
!python setup.py build_ext --inplace
%cd ..

## 4. Sanity check — GPU, espeak, dataset columns

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
!espeak-ng --version

from datasets import load_dataset
_probe = load_dataset('Dmi1tr13/ljspeech-b1', split='train', streaming=True)
print('dataset columns:', list(next(iter(_probe)).keys()))

## 5. Objective eval helper — synth + Whisper WER
Low WER = intelligible, ~100% = garbage. We reuse this after each stage. The fine-tuned model is a standard `VitsModel`, so we load it directly; the tokenizer never changes during fine-tuning, so we always take it from the base LJS model.

In [ ]:
import whisper, jiwer, torch, scipy.io.wavfile
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display

EVAL_TEXT = 'Roger roger. All units, proceed with the mission. Standing by.'
_asr = whisper.load_model('base')
_tok = AutoTokenizer.from_pretrained('kakao-enterprise/vits-ljs')

def evaluate(model_dir, tag, text=EVAL_TEXT):
    model = VitsModel.from_pretrained(model_dir).eval()
    inputs = _tok(text, return_tensors='pt')
    with torch.no_grad():
        wav = model(**inputs).waveform[0].cpu().numpy()
    path = f'demo_{tag}.wav'
    scipy.io.wavfile.write(path, model.config.sampling_rate, wav)
    hyp = _asr.transcribe(path)['text'].lower().strip().strip('.')
    wer = jiwer.wer(text.lower().strip().strip('.'), hyp)
    print(f'[{tag}] WER={wer:.1%}  ->  {hyp!r}')
    display(Audio(path))
    return wer

# Before fine-tuning: the base model should be fully intelligible (~10-20% WER).
evaluate('ylacombe/vits-ljs-with-discriminator', 'pretrained')

## 6. Config builder
We write the recipe's JSON config from a Python dict so both stages share the proven loss weights (DRY, no copy-paste). The key difference from our own loop: `weight_disc` / `weight_gen` / `weight_fmaps` — the adversarial objective.

In [ ]:
import json

BASE_CONFIG = {
    'project_name': 'vits_b1_droid',
    'push_to_hub': False,
    'overwrite_output_dir': True,
    'dataset_name': 'Dmi1tr13/ljspeech-b1',
    'audio_column_name': 'audio',
    'text_column_name': 'text',
    'max_duration_in_seconds': 20,
    'min_duration_in_seconds': 1.0,
    'max_tokens_length': 500,
    'model_name_or_path': 'ylacombe/vits-ljs-with-discriminator',
    'preprocessing_num_workers': 4,
    'do_train': True,
    'gradient_accumulation_steps': 1,
    'gradient_checkpointing': False,
    'per_device_train_batch_size': 16,
    'learning_rate': 2e-5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.99,
    'warmup_ratio': 0.01,
    'do_eval': True,
    'eval_steps': 50,
    'per_device_eval_batch_size': 16,
    'max_eval_samples': 25,
    'do_step_schedule_per_epoch': True,
    # the objective our own loop was missing:
    'weight_disc': 3,
    'weight_fmaps': 1,
    'weight_gen': 1,
    'weight_kl': 1.5,
    'weight_duration': 1,
    'weight_mel': 35,
    'fp16': True,
    'seed': 456,
}

def write_config(name, **overrides):
    cfg = {**BASE_CONFIG, **overrides}
    with open(name, 'w') as f:
        json.dump(cfg, f, indent=2)
    print('wrote', name, '|', {k: cfg[k] for k in ('train_split_name', 'num_train_epochs', 'output_dir')})
    return name

## 7. Stage 1 — small subset (fast sanity check)
Train on the first 300 clips for ~40 epochs (≈760 steps). Goal: confirm the loss moves and the audio becomes intelligible *quickly*, before spending GPU-hours on the full set. The last 500 clips stay held out.

*If you hit CUDA OOM on a T4, lower `per_device_train_batch_size` to 8 (add it as an override below) and set `gradient_accumulation_steps=2`.*

In [ ]:
write_config('config_small.json',
    output_dir='./out_small',
    train_split_name='train[:300]',
    eval_split_name='train[-25:]',
    num_train_epochs=40,
)
!accelerate launch run_vits_finetuning.py config_small.json

In [ ]:
evaluate('./out_small', 'stage1_small')

## 8. Stage 2 — full dataset
The real run: every clip except the held-out 500. `num_train_epochs` scales to your time budget — each epoch ≈ 790 steps at batch 16. Start with ~10 epochs; if the droid voice isn't crisp yet, resume with more.

Eval is on a held-out slice (`train[-500:-475]`), so there is **no test leakage**.

In [ ]:
write_config('config_full.json',
    output_dir='./out_full',
    train_split_name='train[:-500]',
    eval_split_name='train[-500:-475]',
    num_train_epochs=10,
)
!accelerate launch run_vits_finetuning.py config_full.json

In [ ]:
print('=== Before vs after fine-tuning ===')
evaluate('ylacombe/vits-ljs-with-discriminator', 'pretrained')
evaluate('./out_full', 'stage2_full')

## Notes
- **Why this works where our loop didn't:** the recipe adds the discriminator/generator/feature-matching terms (`weight_disc`, `weight_gen`, `weight_fmaps`). Without them the decoder is never trained to produce realistic waveforms → noise.
- **espeak hang:** tokenization happens once during preprocessing (not every training step), so the memory-map leak that hung our baseline doesn't accumulate during training.
- **Save the model:** `out_full/` is a standard `VitsModel` — copy it to Drive, then `VitsModel.from_pretrained('out_full')` anywhere.
- **OOM on T4:** lower `per_device_train_batch_size` to 8 and set `gradient_accumulation_steps=2`.